# The company brain's fresh test: questions nobody has looked at, on documents no earlier test used (A100)

**Why a new test.** The 50,000-document test (`docs/FACTBANK_50K_RESULTS.md`) met its rules. Its answers were then read
closely and eleven things were fixed (`docs/FACTBANK_FIXES.md`): the model's tools, a check before the model changes
the brain's answer, the brain itself (v19), the storage. Those 635 questions are development data now. This notebook
runs the test that decides whether the fixes work, on questions that were never used.

**The questions**
- **463 of the benchmark's own questions** ([EnterpriseRAG-Bench](https://github.com/onyx-dot-app/EnterpriseRAG-Bench)):
  written by its authors in natural language, each with a reference answer and the facts a right answer states. A
  model judges each answer against them; you label a sample of the judge's verdicts afterwards.
- **402 generated questions** with exact answers (lookups, lists, counts, histories, email and chat, questions
  with no answer, and complex questions of two to four steps), drawn with new seeds.
- **The documents:** 50,000 of the benchmark's company: the 684 documents its questions rest on and fresh ones;
  none of the earlier test's other documents.

**Who answers**

| arm | what answers | asked on |
|---|---|---|
| `v19` | the brain alone, no model | every question |
| `reasoner` | the model on the brain: it reads the brain's answer, can use every tool, and its final answer is checked | every question |
| `agent` | the model with search and read only: what an AI agent does without the fact bank | every question |
| `rag` | the model reading 12 passages of one keyword search | every question |
| `reasoner` seeds 1 and 2 | the same again with other sampling seeds | a fixed sample of 100 + 100 |
| `nobrain`, `notools` | the model with the tools but no brain; the model with the brain but no tools | the same sample |

What decides the result was fixed before any model answered: `docs/FACTBANK_FRESH_PLAN.md`. The score cell checks its rules.

**Before you start**
1. Runtime > Change runtime type > **A100 GPU** (80 GB if you can; High-RAM helps).
2. Secrets (the key icon): `HF_TOKEN` is optional (the model is not gated).
3. Run all. Expect **8 to 14 hours** on an A100 80 GB: this test asks about four times as many model answers as the
   last one. It saves to Google Drive (`MyDrive/cie_fresh`) every few minutes, and after a disconnect Run all goes on
   from where it stopped. `LIMIT = 30` gives a quick look in about an hour. `RUN_EXTRA = False` skips the sample
   (rule 8 then stays open) and saves about a quarter of the time.


In [ ]:
#@title Parameters
import os
REPO = "https://github.com/Nikku03/integratedai"
BRANCH = "claude/epic-keller-gevn7j"
MODEL = "auto"                #@param ["auto", "qwen3.8-27b", "qwen3.8-27b-fp8"] {allow-input: true}
EFFORT = "medium"             #@param ["low", "medium", "xhigh"]
RUN_CORE = True               #@param {type:"boolean"}
RUN_EXTRA = True              #@param {type:"boolean"}
RUN_JUDGE = True              #@param {type:"boolean"}
LIMIT = 0                     #@param {type:"integer"}
WORKERS = 0                   #@param {type:"integer"}
SAVE_TO_DRIVE = True          #@param {type:"boolean"}
# MODEL: "auto" = Qwen3.8-27B in bf16 on an 80 GB GPU, Qwen's FP8 copy on a 40 GB one.
# EFFORT: Qwen3.8's reasoning effort. "medium" is the plan's setting.
# RUN_CORE: the arms asked on every question (reasoner, agent, rag). The plan's rules 1 to 7 need them.
# RUN_EXTRA: on the fixed sample, two more seeds of the reasoner and the two ablations. Rule 8 needs them.
# RUN_JUDGE: the model judges the answers to the benchmark's own questions. Rules 4 to 6 need it.
# LIMIT: 0 = every question. A number asks only the first questions of each folder: a quick look, not a decision.
# WORKERS: processes for the brain. 0 picks by this machine's processors and memory.
# SAVE_TO_DRIVE: results go to MyDrive/cie_fresh as each step runs, so a disconnect loses little.
ON_COLAB = os.path.isdir("/content")
WORKDIR = "/content/integratedai/cie" if ON_COLAB else os.environ.get("CIE_COLAB_WORKDIR", os.getcwd())
BENCH = "/content/EnterpriseRAG-Bench-json" if ON_COLAB else os.environ.get("CIE_ERB_ROOT", os.path.join(os.getcwd(), "EnterpriseRAG-Bench"))
BENCH_COMMIT = "d36685e273713975ee20299bbf1ab64165575b3c"  # the benchmark's commit the plan was written with
# the folders stay on this machine's disk: the bank is read at random, which Drive is too slow for
WORK = "/content/fresh/w" if ON_COLAB else os.environ.get("CIE_FRESH_WORK", os.path.join(os.getcwd(), "eval_out", "fresh", "w"))
WORK_BENCH, WORK_SUB, WORK_BENCH_SUB = WORK + "_bench", WORK + "_sub", WORK + "_bench_sub"
FOLDERS = (WORK, WORK_BENCH, WORK_SUB, WORK_BENCH_SUB)
QKEYS = ("questions_sha256_gen", "questions_sha256_bench", "questions_sha256_sub", "questions_sha256_bench_sub")  # each folder's key in EXPECT
INDEX = os.environ.get("B50_INDEX") or os.path.join(os.path.dirname(WORK), "index.json")
OLD = os.path.join(WORKDIR, "docs", "benchmarks", "fresh", "plan", "old_haystack.json.gz")  # the earlier test's documents: none is drawn
# local checks without a GPU: an OpenAI-compatible server stands in for vLLM
BACKEND = os.environ.get("B50_BACKEND", "vllm")
BASE_URL = os.environ.get("B50_BASE_URL", "")
MODEL = os.environ.get("B50_MODEL", MODEL)
LIMIT = int(os.environ.get("B50_LIMIT", LIMIT))
WORKERS = int(os.environ.get("B50_WORKERS", WORKERS))
RUN_MODEL = RUN_CORE or RUN_EXTRA or RUN_JUDGE
RUN_RAG = RUN_REASONER = RUN_MODEL  # the names the set-up cells read
EXPECT = {"dsids_sha256": "284c05ac804aa90b6ea74a470d38375a3845e3440d9481de57e185d789abd7b5", "questions_sha256_gen": "0a3f01cf9e292d9e7d012081c9e3821f0752ded832c14078495996a800402b73", "questions_sha256_bench": "c869799fdfb50e24e78c59f06c0cbad76b8ac9eaac375c97f96869c15a955168", "questions_sha256_sub": "f3f454d30d48851ad76f467bb4603010e15c10349a1111f7037c8481c6d5a000", "questions_sha256_bench_sub": "1748894fbb8f964acc26b6a31b2408fefe318e9f6ee84edfa46e181d276f369a"}  # from the plan's own run of the draw and the questions (docs/FACTBANK_FRESH_PLAN.md)
print("model:", MODEL, "| effort:", EFFORT, "| core arms:", RUN_CORE, "| sample arms:", RUN_EXTRA, "| judge:", RUN_JUDGE,
      "| questions:", LIMIT or "all", "| backend:", BACKEND, "| colab:", ON_COLAB)


In [ ]:
#@title GPU, memory and processors
import os, subprocess
q = subprocess.run("nvidia-smi --query-gpu=name,memory.total,driver_version,compute_cap --format=csv,noheader,nounits", shell=True,
                   capture_output=True, text=True).stdout.strip().splitlines()
print(q[0] if q else "no GPU visible")
GPU_NAME, GPU_MEM_GB, DRIVER_MAJOR, CC = "", 0.0, 0, 0.0
if q:
    f = [x.strip() for x in q[0].split(",")]
    GPU_NAME, GPU_MEM_GB, DRIVER_MAJOR, CC = f[0], int(f[1]) / 1024, int(f[2].split(".")[0]), float(f[3])
if BACKEND == "vllm" and (RUN_RAG or RUN_REASONER):
    if not q:
        raise SystemExit("the model arms need a GPU: Runtime > Change runtime type > A100 GPU, then run all again")
    if CC < 8.0 or GPU_MEM_GB < 35:
        raise SystemExit(f"{GPU_NAME} ({GPU_MEM_GB:.0f} GB, compute capability {CC}) cannot hold the model: choose an A100 or H100")
if MODEL == "auto":
    MODEL = "qwen3.8-27b" if GPU_MEM_GB >= 70 else "qwen3.8-27b-fp8"
MAX_NUM_SEQS = 256 if GPU_MEM_GB >= 70 else 128
OLD_DRIVER = bool(q) and DRIVER_MAJOR < 580  # vLLM's default build needs driver 580 or newer
ram_gb = int(open("/proc/meminfo").read().split()[1]) / 1e6
cpus = os.cpu_count() or 2
if not WORKERS:
    WORKERS = max(2, min(10, cpus - 2, int((ram_gb - 25) // 4)))  # the model needs about 20 GB of memory while it loads
print(f"model {MODEL} | GPU memory {GPU_MEM_GB:.0f} GB | memory {ram_gb:.0f} GB | {cpus} processors -> {WORKERS} brain processes"
      + (f" | driver {DRIVER_MAJOR}: vLLM's CUDA 12.9 build" if OLD_DRIVER else ""))
if ram_gb < 40:
    print("note: little memory; the brain's processes may not fit. Runtime > Change runtime type > High-RAM helps.")

In [ ]:
#@title Clone the repository and install vLLM (a few minutes)
import importlib, os, subprocess, sys
if ON_COLAB and not os.path.isdir(WORKDIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, "/content/integratedai"], check=True)
elif ON_COLAB:
    # an existing checkout moves to the latest commit of the branch: a re-run never tests stale code
    subprocess.run(["git", "fetch", "--depth", "1", "origin", BRANCH], cwd="/content/integratedai", check=True)
    subprocess.run(["git", "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"], cwd="/content/integratedai", check=True)
CODE = subprocess.run(["git", "log", "-1", "--format=%h %ci %s"], cwd=WORKDIR, capture_output=True, text=True).stdout.strip()
print("code at commit:", CODE)
if ON_COLAB and BACKEND == "vllm" and (RUN_RAG or RUN_REASONER):
    # vLLM brings its own torch; this kernel never imports torch, so no restart is needed
    if OLD_DRIVER:
        spec = ["vllm==0.30.0+cu129", "torch==2.13.0+cu129", "--extra-index-url", "https://wheels.vllm.ai/0.30.0/cu129",
                "--extra-index-url", "https://download.pytorch.org/whl/cu129"]
    else:
        spec = ["vllm==0.30.0"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *spec], check=True)
if ON_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", WORKDIR], check=True)  # without optional extras: no second torch
src = os.path.join(WORKDIR, "src")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()
os.chdir(WORKDIR)
if BACKEND == "vllm" and (RUN_RAG or RUN_REASONER):
    r = subprocess.run([sys.executable, "-c", "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, "
                        "'| GPU', torch.cuda.get_device_name(0))"], capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr[-1500:])
    assert r.returncode == 0, "vLLM did not import (output above)"
import cie
print("cie", cie.__version__, "from", os.path.dirname(cie.__file__))

In [ ]:
#@title Keys from Colab secrets (never printed)
import os

def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:  # not on Colab, or no such secret
        return os.environ.get(name)

if v := secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = v
print("HF_TOKEN:", "set" if os.environ.get("HF_TOKEN") else "not set (not needed: the model is not gated)")

In [ ]:
#@title Results folder (Google Drive) and the steps' helpers
import hashlib, json, os, shutil, signal, subprocess, sys, threading, time, uuid
RESULTS = os.environ.get("B50_RESULTS") or os.path.join(os.path.dirname(WORK), "results")
if SAVE_TO_DRIVE and ON_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        RESULTS = "/content/drive/MyDrive/cie_fresh"
    except Exception as e:  # declined or unavailable
        print(f"Google Drive is not available ({type(e).__name__}): results stay on this runtime's disk and are lost with it")
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(os.path.dirname(WORK), exist_ok=True)
RUNS = ("factbank_v19", "reasoner", "agent", "rag", "reasoner.s1", "reasoner.s2", "nobrain", "notools")
ROWS = tuple(f"{r}.jsonl" for r in RUNS) + tuple(f"judge_{r}.jsonl" for r in ("v19", *RUNS[1:]))  # answer and verdict files
ANSWERS = (*ROWS, "arms50k.json")
KEEP = ("haystack.json", "draw_report.json", "banks.json", "questions.jsonl", "questions_report.json", *ANSWERS, "report_fresh.json",
        "report_fresh.md", "labels_to_fill.csv")
SAVE_EVERY = 300  # seconds: a step's answers so far go to Drive this often while it runs

def sha(path):
    return hashlib.sha256(open(path, "rb").read()).hexdigest() if os.path.exists(path) else None

def _lines(path):
    try:
        with open(path, "rb") as f:
            return sum(1 for _ in f)
    except OSError:
        return 0

def _copy(src, dst):
    # written whole (a temporary name, then renamed); a plain copy if the rename fails
    try:
        shutil.copy2(src, dst + ".tmp")
        os.replace(dst + ".tmp", dst)
    except OSError:
        shutil.copy2(src, dst)

def save():
    # every result file of the folders to RESULTS/<folder>/; what cannot be copied now (Drive away) is said, never raised
    for w in FOLDERS:
        out = os.path.join(RESULTS, os.path.basename(w))
        try:
            os.makedirs(out, exist_ok=True)
        except OSError as e:
            print(f"(not saved to Drive now: {type(e).__name__}: {e})", flush=True)
            return
        for f in KEEP:
            src, dst = os.path.join(w, f), os.path.join(out, f)
            try:
                if not os.path.exists(src) or os.path.islink(src):  # the other folders' links to the first one: saved there
                    continue
                if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src) and os.path.getmtime(dst) >= os.path.getmtime(src):
                    continue  # unchanged
                if f in ROWS and _lines(src) < _lines(dst):
                    continue
                _copy(src, dst)
            except OSError as e:
                print(f"(not saved to Drive now: {os.path.basename(w)}/{f}: {type(e).__name__}: {e})", flush=True)

def restore():
    # answers saved by an earlier runtime, put back when they answer the same questions (a row cut off half way by a
    # copy taken while it was written is dropped when the step reads the file, and asked again)
    for w in FOLDERS:
        saved = os.path.join(RESULTS, os.path.basename(w))
        if sha(os.path.join(saved, "questions.jsonl")) is None or sha(os.path.join(saved, "questions.jsonl")) != sha(os.path.join(w, "questions.jsonl")):
            continue
        for f in ANSWERS:
            src, dst = os.path.join(saved, f), os.path.join(w, f)
            if os.path.exists(src) and (not os.path.exists(dst) or os.path.getsize(src) > os.path.getsize(dst)):
                shutil.copy2(src, dst)
                print("restored", os.path.basename(w), f)

def restore_draw():
    # the plan's draw saved by an earlier runtime, so a fresh runtime need not draw again
    saved = os.path.join(RESULTS, os.path.basename(WORK))
    if os.path.exists(os.path.join(WORK, "haystack.json")) or not os.path.exists(os.path.join(saved, "haystack.json")):
        return
    from cie.eval.brain50k import dsids_sha
    h = json.load(open(os.path.join(saved, "haystack.json")))
    if dsids_sha(h["dsids"]) != EXPECT.get("dsids_sha256") or h.get("root") != BENCH or not os.path.exists(os.path.join(saved, "draw_report.json")):
        return
    os.makedirs(WORK, exist_ok=True)
    for f in ("haystack.json", "draw_report.json"):
        shutil.copy2(os.path.join(saved, f), os.path.join(WORK, f))
    if not os.path.lexists(os.path.join(WORK, "index.json")):
        os.symlink(os.path.realpath(INDEX), os.path.join(WORK, "index.json"))
    print("restored the draw from", saved)

def restore_questions():
    # the plan's questions saved by an earlier runtime, so a fresh runtime need not draw them again
    pairs = tuple(zip(FOLDERS, QKEYS))
    if all(os.path.exists(os.path.join(w, "questions.jsonl")) for w, _ in pairs):
        return
    if not all(sha(os.path.join(RESULTS, os.path.basename(w), "questions.jsonl")) == EXPECT.get(k) for w, k in pairs):
        return
    for w, _ in pairs:
        os.makedirs(w, exist_ok=True)
        for f in ("questions.jsonl", "questions_report.json"):
            shutil.copy2(os.path.join(RESULTS, os.path.basename(w), f), os.path.join(w, f))
        for f in ("haystack.json", "index.json", "factbank_v2.sqlite", "banks.json", "draw_report.json"):  # as the questions step links them
            if w != WORK and os.path.exists(os.path.join(WORK, f)) and not os.path.lexists(os.path.join(w, f)):
                os.symlink(os.path.realpath(os.path.join(WORK, f)), os.path.join(w, f))
    print("restored the questions from", RESULTS)

def env():
    return {**os.environ, "PYTHONUNBUFFERED": "1", "PYTHONHASHSEED": "0", "TOKENIZERS_PARALLELISM": "false", "PYTHONPATH": os.path.join(WORKDIR, "src")}

def _stop_step(marker):
    # every process a step started (its tool processes, vLLM's engine), found by the mark in its environment
    pids = []
    for d in os.listdir("/proc"):
        if d.isdigit() and int(d) != os.getpid():
            try:
                if f"B50_STEP={marker}".encode() in open(f"/proc/{d}/environ", "rb").read().split(b"\0"):
                    pids.append(int(d))
            except OSError:
                pass
    for sig in (signal.SIGTERM, signal.SIGKILL):
        for pid in pids:
            try:
                os.kill(pid, sig)
            except OSError:
                pass
        if pids:
            time.sleep(5)
    return pids

def run(args):
    # one step in its own process, its output shown as it comes; its answers so far go to Drive every SAVE_EVERY
    # seconds; processes it leaves behind (a crash that skips their clean-up) are stopped, so the cell always ends
    print("$", " ".join(args), flush=True)
    marker = uuid.uuid4().hex
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors="replace",
                         env={**env(), "B50_STEP": marker}, cwd=WORKDIR)
    stop = threading.Event()

    def show():
        # the output is always read to its end: a full pipe would stop the step
        for line in p.stdout:
            try:
                print(line, end="", flush=True)
            except Exception:  # noqa: BLE001 - a line that cannot be shown is skipped
                pass

    def keep_saving():
        while not stop.wait(SAVE_EVERY):
            try:
                save()
            except Exception as e:  # Drive busy: the next try, or the save after the step
                print(f"(saving to Drive failed: {type(e).__name__}: {e})", flush=True)

    reader, saver = threading.Thread(target=show, daemon=True), threading.Thread(target=keep_saving, daemon=True)
    reader.start()
    saver.start()
    try:
        code = p.wait()
        reader.join(timeout=15)
        if reader.is_alive():
            print(f"(stopped {len(_stop_step(marker))} processes the step left running)", flush=True)
            reader.join(timeout=15)
    except BaseException:
        _stop_step(marker)
        raise
    finally:
        stop.set()
        saver.join()
        try:
            save()
        except Exception as e:
            print(f"(saving to Drive failed: {type(e).__name__}: {e})", flush=True)
    if code != 0:
        raise SystemExit(f"step failed with exit code {code} (output above)")

print("results:", RESULTS)
print("found from an earlier run:", ", ".join(sorted(os.listdir(RESULTS))) or "nothing (a fresh run)")

In [ ]:
#@title Get the benchmark: the repository's JSON records at the plan's commit (about 5 GB, a few minutes)
import os, shutil, subprocess
sources = os.path.join(BENCH, "generated_data", "sources")

def _complete():
    if not os.path.isfile(os.path.join(BENCH, "questions.jsonl")) or not os.path.isdir(os.path.join(BENCH, ".git")):
        return False
    listed = subprocess.run(["git", "ls-files", "generated_data/sources"], cwd=BENCH, capture_output=True, text=True)
    want = sum(1 for f in listed.stdout.splitlines() if f.endswith(".json"))
    have = sum(1 for _, _, fs in os.walk(sources) for f in fs if f.endswith(".json"))
    print(f"benchmark checkout: {have:,} of {want:,} JSON records on disk")
    return listed.returncode == 0 and want > 1000 and have == want

ok = _complete()
if not ok and os.path.isdir(os.path.join(BENCH, ".git")):
    subprocess.run(["git", "checkout", "-f", "HEAD", "--", "."], cwd=BENCH)  # restores files an interrupted checkout left out
    ok = _complete()
if not ok:
    shutil.rmtree(BENCH, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/onyx-dot-app/EnterpriseRAG-Bench", BENCH], check=True)
    ok = _complete()
assert ok, "the benchmark clone is incomplete; run this cell again"
head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=BENCH, capture_output=True, text=True).stdout.strip()
if head != BENCH_COMMIT:
    subprocess.run(["git", "fetch", "--depth", "1", "origin", BENCH_COMMIT], cwd=BENCH, check=True)
    subprocess.run(["git", "checkout", "-q", BENCH_COMMIT], cwd=BENCH, check=True)
    head = BENCH_COMMIT
print("benchmark at commit", head[:12])

In [ ]:
#@title The corpus index: every document's id and file (reads all 512k records once, a few minutes; kept on Drive)
import json, os, shutil
from pathlib import Path
from cie.eval.bench_enterprise import build_index
saved = os.path.join(RESULTS, "index.json")
if not os.path.exists(INDEX) and os.path.exists(saved):
    shutil.copy2(saved, INDEX)
index = build_index(Path(BENCH) / "generated_data" / "sources", cache=Path(INDEX), log=print)
if ON_COLAB and not os.path.exists(saved):
    shutil.copy2(INDEX, saved)
print(f"{len(index):,} documents indexed")

In [ ]:
#@title Draw the 50,000 documents: the ones the benchmark's questions rest on, and fresh ones (a few minutes)
import json, os
from cie.eval.brain50k import dsids_sha
hay = os.path.join(WORK, "haystack.json")
restore_draw()
if not (os.path.exists(hay) and len(json.load(open(hay))["dsids"]) == 50000):
    run([sys.executable, "-u", "-m", "cie.eval.fresh_test", "draw", "--index", INDEX, "--root", BENCH, "--old", OLD, "--out", WORK])
rep = json.load(open(os.path.join(WORK, "draw_report.json")))
got = dsids_sha(json.load(open(hay))["dsids"])
print(f"{rep['documents']:,} documents: {rep['gold_documents']:,} that the benchmark's questions rest on and {rep['added']:,} fresh ones; "
      f"{rep['from_old_haystack']} were in the earlier test's haystack (all of them gold documents)")
print("by system:", rep["by_source"])
print("the same documents as the plan's draw:", got == EXPECT.get("dsids_sha256"), f"({got[:16]})")
assert got == EXPECT.get("dsids_sha256"), "not the plan's documents: the benchmark checkout or the code differs from the plan's"


In [ ]:
#@title Build the fact bank of the 50,000 documents (a few minutes)
import json, os
from pathlib import Path
from cie.eval import brain_test as H
banks = os.path.join(WORK, "banks.json")
fresh = os.path.exists(banks) and json.load(open(banks)).get("factbank_v2", {}).get("haystack_sha256") == H.haystack_sha(Path(WORK))
if not (fresh and os.path.exists(os.path.join(WORK, "factbank_v2.sqlite"))):
    run([sys.executable, "-u", "-m", "cie.eval.brain_test", "build", "--work", WORK])
b = json.load(open(banks))["factbank_v2"]
print(f"bank: {b['sources']:,} documents, {b['facts']:,} facts, {b['bytes'] / 1e9:.2f} GB, built in {b['seconds']:.0f} s, check {b['check_ok']}")

In [ ]:
#@title The questions: generated ones, the benchmark's own, and the fixed sample of each (10 to 20 minutes)
import json, os
restore_questions()
if not all(os.path.exists(os.path.join(w, "questions.jsonl")) for w in FOLDERS):
    run([sys.executable, "-u", "-m", "cie.eval.fresh_test", "questions", "--work", WORK, "--root", BENCH, "--old", OLD])
same = True
for w, key in zip(FOLDERS, QKEYS):
    r = json.load(open(os.path.join(w, "questions_report.json")))
    ok = sha(os.path.join(w, "questions.jsonl")) == EXPECT.get(key)
    same = same and ok
    print(f"{os.path.basename(w)}: {r['questions']} questions {r['by_family']}; the same as the plan's: {ok}")
assert same, "not the plan's questions: the benchmark checkout or the code differs from the plan's"
restore()
save()


In [ ]:
#@title The brain alone (v19): every question, in parallel processes (about 10 to 20 minutes)
# the samples too: the model's arms on a sample read the brain's answers from the sample's own folder
run([sys.executable, "-u", "-m", "cie.eval.brain50k", "ask", "--arm", "v19", "--work", *FOLDERS, "--workers", str(WORKERS)]
    + (["--limit", str(LIMIT)] if LIMIT else []))
save()


In [ ]:
#@title The model: the core arms on every question, the sample's arms, then the judge (one load of the model)
if RUN_MODEL:
    args = [sys.executable, "-u", "-m", "cie.eval.fresh_test", "run", "--work", WORK, WORK_BENCH, "--model", MODEL, "--effort", EFFORT,
            "--workers", str(WORKERS), "--max-num-seqs", str(MAX_NUM_SEQS)]
    args += ([] if RUN_CORE else ["--no-core"]) + ([] if RUN_EXTRA else ["--no-extra"]) + ([] if RUN_JUDGE else ["--no-judge"])
    if LIMIT:
        args += ["--limit", str(LIMIT)]
    if BACKEND != "vllm":
        args += ["--base-url", BASE_URL]
    run(args)
    save()


In [ ]:
#@title Score: generated questions by exact check, the benchmark's by the judge, and the plan's rules
import json, os
from IPython.display import Markdown, display
labels = os.path.join(RESULTS, "labels.csv")  # the sheet you labelled (see the last cell), if it is there
run([sys.executable, "-u", "-m", "cie.eval.fresh_test", "score", "--work", WORK] + (["--labels", labels] if os.path.exists(labels) else []))
save()
display(Markdown(open(os.path.join(WORK, "report_fresh.md")).read()))
if LIMIT:
    print(f"LIMIT = {LIMIT}: a quick look at the first questions, not a decision")


In [ ]:
#@title Look at a few of the benchmark's questions: each arm's answer and the judge's verdict
import json, os, random
def rows(f):
    p = os.path.join(WORK_BENCH, f)
    return {json.loads(l)["id"]: json.loads(l) for l in open(p) if l.strip()} if os.path.exists(p) else {}
qs = rows("questions.jsonl")
ans = {a: rows(f"{f}.jsonl") for a, f in (("v19", "factbank_v19"), ("rag", "rag"), ("agent", "agent"), ("reasoner", "reasoner"))}
jud = {a: rows(f"judge_{a}.jsonl") for a in ans}
done = [i for i in qs if all(i in r for r in ans.values() if r)]
for i in random.Random(3).sample(done, min(4, len(done))):
    q = qs[i]
    print("=" * 100, f"\n{q['kind']}: {q['question']}\nreference: {q['expected']['gold'][:300]}")
    for a, r in ans.items():
        if i in r:
            v = jud[a].get(i, {})
            print(f"  {a:9s} judged {'right' if v.get('correct') else 'wrong' if v.get('correct') is False else 'not judged':10s} {str(r[i]['answer'])[:220]!r}")


In [ ]:
#@title Where the results are, and the sheet to label
import os
sheet = os.path.join(WORK_BENCH, "labels_to_fill.csv")
files = [os.path.join(WORK_BENCH, f"{a}.jsonl") for a in ("reasoner", "agent", "rag") if os.path.exists(os.path.join(WORK_BENCH, f"{a}.jsonl"))]
if files and not os.path.exists(sheet):
    run([sys.executable, "-u", "-m", "cie.eval.judge", "sample", "--questions", os.path.join(WORK_BENCH, "questions.jsonl"),
         "--answers", *files, "--n", "60", "--out", sheet])
    save()
print(f"Results: {RESULTS}" + (" (Google Drive)" if RESULTS.startswith("/content/drive") else " (this runtime's disk)"))
for root, _, fs in os.walk(RESULTS):
    for f in sorted(fs):
        p = os.path.join(root, f)
        print(f"  {os.path.relpath(p, RESULTS):45s} {os.path.getsize(p) / 1e6:8.2f} MB")
print("""
To check the judge: open w_bench/labels_to_fill.csv, write yes or no in the column "correct (yes/no)" for each of the
60 answers (the reference answer is beside each one), save it as labels.csv in the results folder, and run the score
cell again. The judge's verdicts are not in the sheet.
Then paste the report above (or report_fresh.md) into the chat to have it written up.""")
